# Laboratoire du module 1 — Fondamentaux de Spark et Delta Lake

Dans ce laboratoire, vous exécuterez du vrai Spark, observerez comment il exécute votre code et créerez une table Delta que vous pourrez interroger dans le temps et optimiser.

## Avant de commencer
1. Créez un Lakehouse nommé **`training_lh`** (ou réutilisez-en un).
2. Attachez-le à ce notebook (**Add Lakehouse** dans l’Explorateur) et définissez-le comme **par défaut**.
3. Ouvrez la **Spark UI** et **Monitoring** à partir de la barre d’outils du notebook — vous les utiliserez dans le laboratoire 2.


> ### Fonctionnement de ce laboratoire
> - Les cellules **Concept** expliquent l’idée et son importance.
> - Les cellules **Défi** contiennent une structure avec des `À FAIRE` concis à compléter.
> - Développez **Indice** pour obtenir des pistes après avoir tenté le défi.
> - Développez **Solution** pour consulter une approche complète.
> - Les questions de **point de contrôle** comprennent des sections extensibles **Afficher la réponse**.
>
> Exécutez les cellules de haut en bas. Toutes les données sont générées synthétiquement; aucun téléversement n’est requis.

## Configuration &mdash; générer un jeu de données synthétique `trips`
Nous construisons environ 1&nbsp;million de lignes en mémoire avec `spark.range()` et des colonnes dérivées. `spark.range(N)` crée un DataFrame avec une seule colonne `id` distribuée entre les partitions; les appels `withColumn` ajoutent des champs. Exécutez cette cellule telle quelle.

In [ ]:
from pyspark.sql import functions as F

N = 1_000_000
trips = (
    spark.range(N)
    .withColumn("vendorId", (F.col("id") % 3 + 1).cast("int"))
    .withColumn("passengerCount", (F.rand(1) * 5 + 1).cast("int"))
    .withColumn("tripDistance", F.round(F.rand(2) * 20, 2))
    .withColumn("fareAmount", F.round(F.rand(3) * 100 + 3, 2))
    .withColumn("pickupDate", F.expr("date_add(to_date('2024-01-01'), cast(id % 180 as int))"))
    .drop("id")
)
print("Lignes :", trips.count())

## Laboratoire 1 : Votre premier DataFrame

Un **DataFrame** est une table distribuée et en colonnes. Vous décrivez *ce que* vous voulez avec des transformations (`select`, `filter`, `groupBy`); Spark décide *comment* le calculer.

Idée clé à constater ici : **les transformations sont différées**. Construire une chaîne ne fait rien tant que vous n’appelez pas une **action** (`show`, `count`, `collect`, `write`).


### Défi 1.1
Explorez le schéma, puis créez `summary` : conservez les trajets de plus de 5 miles, regroupez par `vendorId`, et retournez le **nombre de trajets** et le **tarif moyen** par fournisseur.

In [ ]:
# Explorer d’abord
trips.printSchema()
trips.show(5)

# À FAIRE : Créer la transformation summary demandée.

# À FAIRE : Afficher le résultat trié par fournisseur.


<details>
<summary><b>Indice</b></summary>

<p>Filtrez avec <code>F.col</code>, regroupez sur <code>vendorId</code>, puis agrégez avec <code>F.count</code> et <code>F.avg</code>. Arrondissez et attribuez un alias à la moyenne avant de l’afficher.</p>

</details>

<details>
<summary><b>Solution</b></summary>

```python
summary = (
    trips.filter(F.col("tripDistance") > 5)
    .groupBy("vendorId")
    .agg(
        F.count("*").alias("trips"),
        F.round(F.avg("fareAmount"), 2).alias("avg_fare"),
    )
)
summary.orderBy("vendorId").show()
```

<p><code>groupBy(...).agg(...)</code> est une transformation <b>large</b> : elle force un shuffle pour que les lignes ayant le même <code>vendorId</code> se retrouvent ensemble. Rien ne s’est exécuté avant <code>.show()</code>.</p>

</details>

## Laboratoire 2 : Évaluation différée et DAG

Comme Spark voit votre chaîne **entière** avant de l’exécuter, l’optimiseur **Catalyst** peut élaguer des colonnes et pousser les filtres vers le bas. Une **action** compile le plan en un **DAG** de stages; chaque transformation *large* (shuffle) démarre un nouveau **stage**.

`DataFrame.explain(True)` affiche les quatre phases du plan : **parsed &rarr; analyzed &rarr; optimized &rarr; physical**.


### Défi 2.1
Créez une chaîne en plusieurs étapes nommée `plan`, sans appeler d’action, et inspectez son plan. Déclenchez ensuite l’exécution et utilisez Spark UI -> Jobs -> Stages pour repérer la limite du shuffle.

In [ ]:
# À FAIRE : Créer le plan de transformation paresseux.
plan = (
    trips
    # À FAIRE
    # À FAIRE
)

# À FAIRE : Imprimer le plan sans déclencher de job.

# À FAIRE : Déclencher le plan, puis inspecter ses stages dans la Spark UI.

<details>
<summary><b>Indice</b></summary>

<p>Enchaînez un filtre, un regroupement et un décompte dans <code>plan</code>. Inspectez-le avec la méthode du plan étendu avant d’appeler une action. Puis affichez le résultat et repérez la limite de l’échange dans les stages de la Spark UI.</p>

</details>

<details>
<summary><b>Solution</b></summary>

```python
plan = (
    trips.filter(F.col("fareAmount") > 50)
    .groupBy("vendorId")
    .count()
)
plan.explain(True)   # toujours aucun job dans la Spark UI
plan.show()

# Spark UI -> Jobs -> (votre job) -> Stages:
#   Stage 1 lit, filtre et agrège partiellement
#   Limite du shuffle
#   Stage 2 termine l’agrégation
```

<p>Cherchez un nœud <code>Exchange</code> dans le plan physique; c’est le shuffle et la limite entre les stages.</p>

</details>

### Du DataFrame à une table Lakehouse gérée
Lorsque vous appelez `df.write.format("delta").saveAsTable("trips")`, deux choses se produisent en même temps :
1. Les **données** sont écrites sous forme de fichiers Delta (Parquet + `_delta_log/`) sous `Tables/` dans le Lakehouse sur OneLake.
2. La table est **enregistrée dans le metastore**, ce qui la rend immédiatement accessible depuis Spark, le **SQL analytics endpoint (T-SQL)** et **Power BI (Direct Lake)** — sans copies.

C’est ce que signifie « table gérée » : *données dans OneLake + entrée de catalogue*. Gardez cela en tête lorsque vous créerez la table Delta ci-dessous.


## Laboratoire 3 : Créer une table Delta

**Delta Lake** = fichiers de données Parquet **+** journal de transactions (`_delta_log/`). Chaque écriture est un **commit atomique et versionné**. `DESCRIBE HISTORY` affiche le journal.


### Défi 3.1
Écrivez `trips` comme table **Delta** nommée `trips`, en l’écrasant si elle existe, puis affichez son historique (version, horodatage, opération).

In [ ]:
# Réinitialiser la table afin que cette écriture crée toujours la version 0.
spark.sql("DROP TABLE IF EXISTS trips")

# À FAIRE : Persister le DataFrame trips comme table gérée demandée.

# À FAIRE : Afficher l’historique de la table.


<details>
<summary><b>Indice</b></summary>

<p>Utilisez le writer du DataFrame avec le format Delta, le mode overwrite et l’enregistrement comme table gérée. Interrogez <code>DESCRIBE HISTORY</code> et sélectionnez les colonnes demandées.</p>

</details>

<details>
<summary><b>Solution</b></summary>

```python
trips.write.format("delta").mode("overwrite").saveAsTable("trips")
spark.sql("DESCRIBE HISTORY trips").select("version", "timestamp", "operation").show(truncate=False)
```

<p>Explorez <b><code>Tables/trips</code></b> et son dossier <b><code>_delta_log/</code></b> dans l’Explorateur Lakehouse; les commits JSON sont la source de vérité.</p>

</details>

### Défi 3.2
Exécutez un `UPDATE` (un nouveau commit atomique) qui augmente `fareAmount` de 10 % pour `vendorId = 1`, puis revérifiez l’historique; une nouvelle version devrait apparaître.

In [ ]:
# À FAIRE : Appliquer la mise à jour demandée à la table Delta.

# À FAIRE : Afficher le dernier historique de la table.

<details>
<summary><b>Indice</b></summary>

<p>Utilisez Spark SQL pour mettre à jour les lignes du fournisseur correspondant et arrondir le tarif ajusté. Interrogez ensuite <code>DESCRIBE HISTORY</code> pour vérifier qu’une nouvelle opération a été enregistrée.</p>

</details>

<details>
<summary><b>Solution</b></summary>

```python
spark.sql("UPDATE trips SET fareAmount = ROUND(fareAmount * 1.1, 2) WHERE vendorId = 1")
spark.sql("DESCRIBE HISTORY trips").select("version", "operation").show()
```

</details>

## Laboratoire 4 : Voyage temporel, compaction et OPTIMIZE

Comme les commits sont versionnés, vous pouvez **voyager dans le temps**. Et comme les écritures en streaming ou de petite taille créent **beaucoup de petits fichiers**, `OPTIMIZE` les compacte; `ZORDER` regroupe les valeurs pour l’élagage de fichiers (data skipping); `VACUUM` supprime les fichiers périmés.


### Défi 4.1
Interrogez la **version 0** de `trips` (avant la mise à jour du laboratoire 3) et comparez le tarif moyen par fournisseur avec la version actuelle.

In [ ]:
# À FAIRE : Comparer les agrégats historiques et actuels demandés.

<details>
<summary><b>Indice</b></summary>

<p>Exécutez une agrégation SQL contre <code>VERSION AS OF 0</code> et la même agrégation contre la table actuelle. Regroupez et triez par fournisseur afin que les résultats soient faciles à comparer.</p>

</details>

<details>
<summary><b>Solution</b></summary>

```python
spark.sql('''
  SELECT vendorId, ROUND(AVG(fareAmount),2) AS avg_fare
  FROM trips VERSION AS OF 0
  GROUP BY vendorId ORDER BY vendorId
''').show()

spark.sql('''
  SELECT vendorId, ROUND(AVG(fareAmount),2) AS avg_fare
  FROM trips
  GROUP BY vendorId ORDER BY vendorId
''').show()
```

<p><code>vendorId</code> 1 devrait être environ 10 % plus élevé dans la version actuelle. <code>TIMESTAMP AS OF '2024-01-01T00:00:00'</code> fonctionne aussi.</p>

</details>

### Défi 4.2
Créez de nombreux petits fichiers en ajoutant 100 fois une tranche à `trips_small`, puis **compactez** la table avec `OPTIMIZE` et `ZORDER BY (vendorId)`.


In [ ]:
%run module_1_helpers


In [ ]:
# Réinitialiser la table pour rendre les statistiques comparables à chaque exécution.
spark.sql("DROP TABLE IF EXISTS trips_small")

small = trips.limit(200000)
for i in range(100):
    small.write.format("delta").mode("append").saveAsTable("trips_small")


In [ ]:
before = table_file_stats("trips_small")
print_stats(before, "AVANT OPTIMIZE")


In [ ]:
# À FAIRE : Exécuter OPTIMIZE avec ZORDER BY (vendorId).


In [ ]:
after = table_file_stats("trips_small")
print_stats(after, "APRÈS OPTIMIZE")


<details>
<summary><b>Indice</b></summary>

Utilisez <code>OPTIMIZE</code> avec <code>ZORDER BY (vendorId)</code>.

</details>


<details>
<summary><b>Solution</b></summary>

```python
spark.sql("OPTIMIZE trips_small ZORDER BY (vendorId)")
```

</details>


## Point de contrôle

Répondez de mémoire, puis affichez la réponse.

**1. Qu’est-ce qui déclenche un job Spark, et qu’est-ce qui crée un nouveau stage ?**

<details>
<summary><b>Afficher la réponse</b></summary>

<p>Une <b>action</b> (<code>count</code>, <code>show</code>, <code>collect</code>, <code>write</code>) déclenche un job. Une <b>transformation large</b> (un shuffle, p. ex. <code>groupBy</code>, <code>join</code>, <code>distinct</code>) crée une nouvelle limite de <b>stage</b>.</p>

</details>

**2. Quelle est la différence entre une transformation étroite et une transformation large ?**

<details>
<summary><b>Afficher la réponse</b></summary>

<p><b>Étroite</b> (<code>filter</code>, <code>map</code>, <code>withColumn</code>) : chaque partition de sortie dépend d’une seule partition d’entrée, sans déplacement de données. <b>Large</b> (<code>groupBy</code>, <code>join</code>) : les partitions de sortie dépendent de plusieurs partitions d’entrée, ce qui exige un <b>shuffle</b> sur le réseau.</p>

</details>

**3. Où Delta stocke-t-il la vérité sur les fichiers qui appartiennent à une table ?**

<details>
<summary><b>Afficher la réponse</b></summary>

<p>Dans le dossier <b><code>_delta_log/</code></b> : des commits JSON ordonnés et des checkpoints périodiques. La liste physique des fichiers ne fait <i>pas</i> autorité; le journal, oui.</p>

</details>

**4. À quoi servent `OPTIMIZE` et `ZORDER BY` ?**


<details>
<summary><b>Afficher la réponse</b></summary>

<p><code>OPTIMIZE</code> compacte de <b>nombreux petits fichiers</b> en un plus petit nombre de gros fichiers. <code>ZORDER BY</code> regroupe les données selon les colonnes fréquemment filtrées afin d’améliorer le <i>data skipping</i>.</p>

</details>


### Nettoyage (facultatif)

In [ ]:
# spark.sql("DROP TABLE IF EXISTS trips")
# spark.sql("DROP TABLE IF EXISTS trips_small")